# Reproduce Results

This notebook **does not train anything**.  It loads the two models that `train_models.ipynb` saved to `./models/` and produces the metrics dataframe required by the deliverable: ROC-AUC, precision and recall on the train / validation / test splits, for both the baseline and the improved model.

It should run end-to-end in well under 10 minutes.

**Data location.** Reads from `$HOME/Datasets/QuoraQuestionPairs/quora_data.csv` as required by the deliverable.

In [ ]:
import os
import joblib
import numpy as np
import pandas as pd

import utils

MODELS_DIR = './models'
BASELINE_PATH = os.path.join(MODELS_DIR, 'baseline.joblib')
IMPROVED_PATH = os.path.join(MODELS_DIR, 'improved.joblib')

assert os.path.exists(BASELINE_PATH), 'Run train_models.ipynb first.'
assert os.path.exists(IMPROVED_PATH), 'Run train_models.ipynb first.'

## 1. Reload data and reproduce the official split

In [ ]:
df = utils.load_quora_dataframe()
train_df, val_df, test_df = utils.split_quora(df)
print('train:', train_df.shape, 'val:', val_df.shape, 'test:', test_df.shape)

## 2. Load the trained models from disk

In [ ]:
baseline = joblib.load(BASELINE_PATH)
improved = joblib.load(IMPROVED_PATH)
print('baseline keys:', list(baseline.keys()))
print('improved keys:', list(improved.keys()))

## 3. Predict and score

We compute ROC-AUC, precision and recall (with a 0.5 decision threshold) on every split for both models.

In [ ]:
splits = {'train': train_df, 'val': val_df, 'test': test_df}

rows = []

# --- Baseline ---------------------------------------------------------
for split_name, split_df in splits.items():
    proba, _ = utils.predict_baseline(baseline['vec'], baseline['clf'], split_df)
    m = utils.evaluate(split_df['is_duplicate'].values, proba)
    rows.append({'model': 'baseline (TF-IDF + LogReg)', 'split': split_name, **m})

# --- Improved --------------------------------------------------------
for split_name, split_df in splits.items():
    X = utils.build_feature_matrix(split_df, improved['tfidf_scratch'], improved['tfidf_sklearn'])
    proba = improved['model'].predict_proba(X)[:, 1]
    m = utils.evaluate(split_df['is_duplicate'].values, proba)
    rows.append({'model': 'improved (features + XGBoost)', 'split': split_name, **m})

results = pd.DataFrame(rows)
results = results[['model', 'split', 'roc_auc', 'precision', 'recall']]
results

## 4. Summary view

In [ ]:
summary = results.set_index(['model', 'split']).round(4)
summary

In [ ]:
# Optional: feature importances (informational, not part of the metric report)
imp = pd.Series(improved['model'].feature_importances_, index=improved['feature_names'])
imp.sort_values(ascending=False)